# benchmark_chrombpnet

ChromBPNet / CAPY ATAC-seq benchmark notebook. Loads pre-computed evaluation outputs per fold and produces aggregated count scatter plots (Fig 1c style), JSD histogram comparisons (Fig 1d style, aggregated over folds), fold-level metric line plots, and numerical CSVs.

In [ ]:
from __future__ import annotations

from pathlib import Path
import csv
import sys
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

warnings.filterwarnings("ignore", category=RuntimeWarning)
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "examples" / "atac" / "file_config.py").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing examples/atac/file_config.py")


REPO_ROOT = find_repo_root()
EXAMPLES_ATAC_DIR = REPO_ROOT / "examples" / "atac"
for _p in [REPO_ROOT, EXAMPLES_ATAC_DIR]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from file_config import AtacFoldFilesConfig

pd.set_option("display.max_columns", None)

In [ ]:
# -----------------------------
# User config
# -----------------------------
proj_dir = Path("/grid/koo/home/shared/capybara/atac")
cell_type = "K562"
folds = [1, 2, 3, 4, 5]
rc_augmented = True
split = "test"

# One timestamp per model, reused across all folds.
# Update timestamps to the runs you want to compare.
model_specs = [
    {"label": "CAPY", "model_name": "capy", "timestamp": "FILL_IN", "color": "#2166ac"},
    # {"label": "CAPY v2", "model_name": "capy", "timestamp": "FILL_IN", "color": "#d6604d"},
]

outdir = REPO_ROOT / "examples" / "atac" / "results" / "benchmark"
outdir.mkdir(parents=True, exist_ok=True)
outdir

In [ ]:
# Color sanity-check plot
x = np.linspace(0, 10, 200)
fig, ax = plt.subplots(figsize=(7, 3))
for i, spec in enumerate(model_specs):
    ax.plot(x, np.sin(x + i * 0.6) + i * 0.15, label=spec["label"], color=spec["color"], linewidth=3)
ax.set_title("Model color preview")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------
# General helpers
# -----------------------------
model_specs_df = pd.DataFrame(model_specs).copy()
rc_suffix = "_rc" if rc_augmented else ""


def model_name_join(df: pd.DataFrame = model_specs_df) -> str:
    return "_".join(df["label"].tolist())


def get_fold_configs(model_name: str, timestamp: str) -> list[AtacFoldFilesConfig]:
    return [
        AtacFoldFilesConfig.create(
            proj_dir=proj_dir,
            cell_type=cell_type,
            fold=int(fold),
            model_name=model_name,
            timestamp=timestamp,
        )
        for fold in folds
    ]


def metrics_summary_path(files: AtacFoldFilesConfig) -> Path:
    return files.eval_dir / f"{cell_type}_metrics_summary{rc_suffix}_{split}.csv"


def jsd_array_path(files: AtacFoldFilesConfig, jsd_type: str) -> Path:
    return files.eval_dir / f"{cell_type}_{jsd_type}{rc_suffix}_{split}.npy"


def count_npy_path(files: AtacFoldFilesConfig, kind: str) -> Path:
    return files.eval_dir / f"{cell_type}_log_{kind}_counts{rc_suffix}_{split}.npy"


def save_figure(fig, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight", pad_inches=0.08)
    plt.show()
    print("saved:", path)


def fold_mean_std(values: list[float]) -> tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float("nan"), float("nan")
    if arr.size == 1:
        return float(arr[0]), float("nan")
    return float(np.mean(arr)), float(np.std(arr, ddof=1))


def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(rows[0].keys()) if rows else []
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print("saved:", path)

In [ ]:
model_specs_df

In [ ]:
# -----------------------------
# Load pre-computed summary metrics (one row per model x fold)
# -----------------------------
summary_rows = []
for _, spec in model_specs_df.iterrows():
    fold_cfgs = get_fold_configs(spec["model_name"], spec["timestamp"])
    for cfg in fold_cfgs:
        path = metrics_summary_path(cfg)
        if not path.exists():
            raise FileNotFoundError(f"Missing summary CSV: {path}")
        row = pd.read_csv(path).iloc[0].to_dict()
        row["model_label"] = spec["label"]
        row["color"] = spec["color"]
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("summary_df shape:", summary_df.shape)
summary_df[["model_label", "fold", "count_pearson", "count_r2", "jsd", "profile_pearson"]].head(10)

In [ ]:
# -----------------------------
# Load JSD arrays; concatenate across folds
# Skipped gracefully if not saved during evaluation.
# -----------------------------
JSD_TYPES = ["jsd_pred", "jsd_shuffled", "jsd_mean", "jsd_pseudorep"]
jsd_arrays: dict[tuple[str, str], np.ndarray] = {}
have_jsd = True

for _, spec in model_specs_df.iterrows():
    if not have_jsd:
        break
    fold_cfgs = get_fold_configs(spec["model_name"], spec["timestamp"])
    for jsd_type in JSD_TYPES:
        per_fold = []
        for cfg in fold_cfgs:
            p = jsd_array_path(cfg, jsd_type)
            if not p.exists():
                warnings.warn(f"Missing JSD array: {p} — Fig 1d will be skipped")
                have_jsd = False
                break
            per_fold.append(np.load(p))
        if have_jsd:
            jsd_arrays[(spec["label"], jsd_type)] = np.concatenate(per_fold)

print(f"JSD arrays loaded: {have_jsd}")
if have_jsd:
    first = model_specs_df.iloc[0]["label"]
    for jt in JSD_TYPES:
        arr = jsd_arrays[(first, jt)]
        print(f"  {jt}: n={arr.size}, median={np.nanmedian(arr):.4f}")

In [ ]:
# -----------------------------
# Load count arrays; concatenate across folds
# Only available when evaluate.py was run with --save_predictions.
# -----------------------------
count_true: dict[tuple[str, int], np.ndarray] = {}
count_pred: dict[tuple[str, int], np.ndarray] = {}
have_counts = True

for _, spec in model_specs_df.iterrows():
    if not have_counts:
        break
    fold_cfgs = get_fold_configs(spec["model_name"], spec["timestamp"])
    for cfg in fold_cfgs:
        tp = count_npy_path(cfg, "true")
        pp = count_npy_path(cfg, "pred")
        if not tp.exists() or not pp.exists():
            warnings.warn(f"Missing count .npy for {spec['label']} fold {cfg.fold} — Fig 1c will be skipped")
            have_counts = False
            break
        count_true[(spec["label"], cfg.fold)] = np.load(tp).reshape(-1).astype(float)
        count_pred[(spec["label"], cfg.fold)] = np.load(pp).reshape(-1).astype(float)

print(f"Count arrays loaded: {have_counts}")
if have_counts:
    first = model_specs_df.iloc[0]["label"]
    n = sum(count_true[(first, f)].size for f in folds)
    print(f"  {first}: total peaks = {n:,}")


def aggregate_counts(label: str) -> tuple[np.ndarray, np.ndarray]:
    return (
        np.concatenate([count_true[(label, f)] for f in folds]),
        np.concatenate([count_pred[(label, f)] for f in folds]),
    )

In [ ]:
# -----------------------------
# Numerical results CSVs
# -----------------------------
count_metric_cols = ["count_pearson", "count_spearman", "count_r2"]
profile_metric_cols = ["jsd", "profile_pearson", "profile_spearman"]

count_agg_rows = []
profile_agg_rows = []

for _, spec in model_specs_df.iterrows():
    label = spec["label"]
    msub = summary_df[summary_df["model_label"] == label]

    crow = {"model": label, "model_name": spec["model_name"], "timestamp": spec["timestamp"]}
    for col in count_metric_cols:
        m, s = fold_mean_std(msub[col].tolist())
        crow[f"{col}_fold_mean"] = m
        crow[f"{col}_fold_std"] = s
    count_agg_rows.append(crow)

    prow = {"model": label, "model_name": spec["model_name"], "timestamp": spec["timestamp"]}
    for col in profile_metric_cols:
        m, s = fold_mean_std(msub[col].tolist())
        prow[f"{col}_fold_mean"] = m
        prow[f"{col}_fold_std"] = s
    for jsd_col in ["jsd_pred_median", "jsd_shuffled_median", "jsd_mean_median", "jsd_pseudorep_median"]:
        if jsd_col in msub.columns:
            m, s = fold_mean_std(msub[jsd_col].tolist())
            prow[f"{jsd_col}_fold_mean"] = m
            prow[f"{jsd_col}_fold_std"] = s
    profile_agg_rows.append(prow)

write_csv(outdir / f"count_metrics_aggregate_{cell_type}.csv", count_agg_rows)
write_csv(outdir / f"profile_metrics_aggregate_{cell_type}.csv", profile_agg_rows)

In [ ]:
# -----------------------------
# Count metrics summary (fold mean ± std)
# -----------------------------
print(f"=== Count metrics ({cell_type}, split={split}) ===")
for _, spec in model_specs_df.iterrows():
    label = spec["label"]
    msub = summary_df[summary_df["model_label"] == label]
    print(f"\n{label}")
    for col in count_metric_cols:
        m, s = fold_mean_std(msub[col].tolist())
        print(f"  {col}: {m:.4f} ± {s:.4f}")

In [ ]:
# -----------------------------
# Profile metrics summary (fold mean ± std)
# -----------------------------
print(f"=== Profile metrics ({cell_type}, split={split}) ===")
for _, spec in model_specs_df.iterrows():
    label = spec["label"]
    msub = summary_df[summary_df["model_label"] == label]
    print(f"\n{label}")
    for col in profile_metric_cols:
        m, s = fold_mean_std(msub[col].tolist())
        print(f"  {col}: {m:.4f} ± {s:.4f}")
    for jsd_col in ["jsd_pred_median", "jsd_shuffled_median", "jsd_mean_median", "jsd_pseudorep_median"]:
        if jsd_col in msub.columns:
            m, s = fold_mean_std(msub[jsd_col].tolist())
            print(f"  {jsd_col}: {m:.4f} ± {s:.4f}")

In [ ]:
# -----------------------------
# Aggregated count scatter — Fig 1c style (2D histogram)
# Requires evaluate.py --save_predictions; skipped otherwise.
# -----------------------------
if not have_counts:
    print("Skipping Fig 1c: run evaluate.py with --save_predictions to enable this plot.")
else:
    n_models = len(model_specs_df)
    panel_size = 4.5
    fig, axes = plt.subplots(1, n_models, figsize=(panel_size * n_models, panel_size), dpi=200, squeeze=False)
    for i, (_, spec) in enumerate(model_specs_df.iterrows()):
        ax = axes[0, i]
        tl, pl = aggregate_counts(spec["label"])
        finite = np.isfinite(tl) & np.isfinite(pl)
        r, _ = pearsonr(tl[finite], pl[finite])
        h = ax.hist2d(tl[finite], pl[finite], bins=100, cmap="Blues", norm=matplotlib.colors.LogNorm())
        fig.colorbar(h[3], ax=ax, label="Count")
        lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lims, lims, "k--", linewidth=0.8, alpha=0.6)
        ax.set_xlabel("Observed log counts", fontsize=9)
        if i == 0:
            ax.set_ylabel("Predicted log counts", fontsize=9)
        ax.set_title(f"{spec['label']} — Pearson r = {r:.3f}", fontsize=9)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(length=2, labelsize=7)
    fig.suptitle(f"{cell_type} — aggregated over folds {folds}", fontsize=10, y=1.01)
    fig.tight_layout()
    save_figure(fig, outdir / f"count_scatter_agg_{cell_type}.pdf")

In [ ]:
# -----------------------------
# JSD histogram comparison — Fig 1d style, aggregated over folds
# Baselines (shuffled, mean, pseudorep) come from the first model's arrays
# since they depend only on observed data, not model predictions.
# Skipped if JSD .npy files were not saved during evaluation.
# -----------------------------
if not have_jsd:
    print("Skipping Fig 1d: JSD arrays not available.")
else:
    bins = np.linspace(0, 1, 101)
    first = model_specs_df.iloc[0]["label"]

    baselines = [
        (jsd_arrays[(first, "jsd_shuffled")], "#555555", "Shuffled obs. vs. obs."),
        (jsd_arrays[(first, "jsd_mean")],     "#2ca25f", "Mean profile vs. obs."),
        (jsd_arrays[(first, "jsd_pseudorep")],"#d73027", "Pseudoreplicate upper bound"),
    ]

    fig, ax = plt.subplots(figsize=(8, 5), dpi=200)

    for arr, color, label in baselines:
        finite = arr[np.isfinite(arr)]
        ax.hist(finite, bins=bins, color=color, alpha=0.45,
                label=f"{label} (median={np.nanmedian(arr):.3f})")
        ax.axvline(np.nanmedian(arr), color=color, linestyle="--", linewidth=1.2)

    for _, spec in model_specs_df.iterrows():
        arr = jsd_arrays[(spec["label"], "jsd_pred")]
        finite = arr[np.isfinite(arr)]
        ax.hist(finite, bins=bins, color=spec["color"], alpha=0.55,
                label=f"{spec['label']} pred. vs. obs. (median={np.nanmedian(arr):.3f})")
        ax.axvline(np.nanmedian(arr), color=spec["color"], linestyle="-", linewidth=1.5)

    ax.set_xlabel("Jensen-Shannon Distance", fontsize=10)
    ax.set_ylabel("Number of peaks", fontsize=10)
    ax.set_title(f"{cell_type} — JSD distributions aggregated over folds {folds}", fontsize=10)
    ax.legend(loc="upper right", fontsize=7, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(length=2, labelsize=8)
    fig.tight_layout()
    save_figure(fig, outdir / f"jsd_hist_agg_{cell_type}.pdf")

In [ ]:
# -----------------------------
# Fold-level metrics line plots
# -----------------------------
metric_rows = [
    ("jsd",              "Profile JSD (mean, ↓)"),
    ("profile_pearson",  "Profile Pearson (mean, ↑)"),
    ("profile_spearman", "Profile Spearman (mean, ↑)"),
    ("count_pearson",    "Count Pearson (↑)"),
    ("count_spearman",   "Count Spearman (↑)"),
    ("count_r2",         "Count R2 (↑)"),
]
if "jsd_pred_median" in summary_df.columns:
    metric_rows += [
        ("jsd_pred_median",      "JSD pred median (↓)"),
        ("jsd_mean_median",      "JSD mean baseline (↓)"),
        ("jsd_pseudorep_median", "JSD pseudorep bound (↓)"),
    ]

n_metrics = len(metric_rows)
n_cols = 3
n_rows_grid = (n_metrics + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows_grid, n_cols,
    figsize=(3.5 * n_cols, 2.5 * n_rows_grid),
    dpi=200,
    sharex=True,
)
axes_flat = np.atleast_1d(axes).reshape(-1)

for idx, (col, title) in enumerate(metric_rows):
    ax = axes_flat[idx]
    for _, spec in model_specs_df.iterrows():
        msub = summary_df[summary_df["model_label"] == spec["label"]].sort_values("fold")
        ax.plot(
            msub["fold"].tolist(), msub[col].tolist(),
            marker="o", linewidth=1.4, markersize=3.5,
            color=spec["color"], label=spec["label"],
        )
    ax.set_title(title, fontsize=8)
    ax.set_xticks(folds)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(length=2, labelsize=7)
    if idx >= n_metrics - n_cols:
        ax.set_xlabel("Fold", fontsize=7)

for ax in axes_flat[n_metrics:]:
    ax.set_visible(False)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, fontsize=8, ncol=len(labels),
           loc="upper center", bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(rect=(0, 0, 1, 0.96), w_pad=1.0, h_pad=1.2)
save_figure(fig, outdir / f"cmp_fold_metrics_{cell_type}.pdf")